# Hansen Ch.25 Binary Choice

**Chapter 25 Binary Choice**（书稿 PDF 约 **p836–838**，习题 **25.1–25.19**）

理论推导（翻转 $Y$、单位缩放、$h/H$、score/Hessian、IV probit、异方差识别）见同目录  
`Hansen_Ch25_Exercises_Solutions.md`。

本 notebook 实现 **probit MLE（Fisher scoring）** 与 CPS 实证 **25.15–25.19**。

> **写给只学过李子奈/陈强的同学：**
> - **LPM**：$\mathbb{E}[Y\mid X]=X'\beta$，OLS 即可，但 $\hat P$ 可能越出 $[0,1]$，且误差必然异方差。
> - **Probit**：潜变量 $Y^*=X'\beta+e$，$e\sim N(0,1)$，$Y=1\{Y^*>0\}$ ⇒ $P(Y=1\mid X)=\Phi(X'\beta)$。
> - **系数尺度**：$\mathrm{Var}(e)=1$ 是识别用的标准化，**不宜直接与 OLS 系数比大小**；报告 **平均边际效应 AME** 更易解释。
> - **Fisher scoring**：用信息矩阵 $\sum_i X_iX_i' \phi_i^2/(\Phi_i(1-\Phi_i))$ 做 Newton 步，比纯 BFGS 在稀有事件（工会率 ~2%）下更稳。


## 0. 工具函数：probit MLE + sandwich SE + AME

**似然**（观测 $i$）：

$$
\ell_i(\beta)=Y_i\log\Phi(X_i'\beta)+(1-Y_i)\log\bigl(1-\Phi(X_i'\beta)\bigr).
$$

**Score**（对 $\beta$）：

$$
s_i = X_i\cdot\frac{(Y_i-\Phi_i)\phi_i}{\Phi_i(1-\Phi_i)}.
$$

**Fisher 信息权重**（$H_{\mathrm{probit}}$ 的期望形式）：

$$
w_i = \frac{\phi_i^2}{\Phi_i(1-\Phi_i)},\qquad
\mathcal{I}_n = \sum_i X_i X_i' w_i.
$$

**迭代**：$\beta\leftarrow\beta+\mathcal{I}_n^{-1}\sum_i s_i$。

**稳健 SE**：sandwich $\widehat V = \mathcal{I}_n^{-1}\bigl(\sum_i s_i s_i'\bigr)\mathcal{I}_n^{-1}$。

**连续回归元 AME**：$\widehat{\mathrm{AME}}_j = \bar\phi\,\hat\beta_j$，其中 $\bar\phi=n^{-1}\sum_i\phi(X_i'\hat\beta)$。


In [ ]:
# Hansen Ch.25 Binary Choice — probit MLE（详尽注释）
# 数据：cps09mar（本地 hansen/econometrics/data，已被 gitignore）

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm

# 仓库根目录相对本 notebook 的路径（docs/ch25/ → 上两级）
ROOT = Path("../..") / "hansen" / "econometrics" / "data"  # relative to docs/chXX/
def probit_mle(y, X, maxiter=50, tol=1e-10, start_scale=0.1):
    """
    Probit 极大似然估计（Fisher scoring）。

    参数
    ----
    y : (n,) 二元因变量 0/1
    X : (n, k) 设计矩阵（含截距列）
    maxiter, tol : 迭代控制
    start_scale : 用 LPM 系数乘该因子作初值（probit 尺度常比 LPM 小）

    返回
    ----
    beta, se_robust, ame, loglik, n_iter
    """
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)
    n, k = X.shape

    # ---- 初值：LPM 的 OLS，再缩小到 probit 尺度附近 ----
    # 本科对照：线性概率模型 beta_LPM ≈ 边际效应；probit 系数 ≈ AME / phi(0) 量级
    beta = np.linalg.lstsq(X, y, rcond=None)[0] * start_scale

    for it in range(maxiter):
        # 线性指数 eta = X'beta；截断避免 Phi 下溢到 0 导致 log 爆炸
        eta = np.clip(X @ beta, -8.0, 8.0)
        Phi = np.clip(norm.cdf(eta), 1e-12, 1.0 - 1e-12)
        phi = norm.pdf(eta)

        # score 权重：(Y-Phi)*phi / (Phi*(1-Phi))
        # Fisher 权重：phi^2 / (Phi*(1-Phi))  —— 正确设定下 E[s s'] 的主部
        denom = Phi * (1.0 - Phi)
        score_w = (y - Phi) * phi / denom          # (n,) 标量 score 因子
        fish_w = (phi ** 2) / denom                # (n,) Fisher 权重

        # g = sum_i X_i * score_w_i
        # H ≈ X' diag(fish_w) X  （信息矩阵，正定方向）
        g = X.T @ score_w
        H = X.T @ (X * fish_w[:, None])

        step = np.linalg.solve(H, g)
        beta = beta + step
        if np.max(np.abs(step)) < tol:
            break

    # ---- 收敛后的 SE 与 AME ----
    eta = np.clip(X @ beta, -8.0, 8.0)
    Phi = np.clip(norm.cdf(eta), 1e-12, 1.0 - 1e-12)
    phi = norm.pdf(eta)
    denom = Phi * (1.0 - Phi)
    score_w = (y - Phi) * phi / denom
    fish_w = (phi ** 2) / denom

    bread = X.T @ (X * fish_w[:, None])           # I_n
    # meat = sum s_i s_i' ，s_i = X_i * score_w_i
    meat = X.T @ (X * (score_w ** 2)[:, None])
    Binv = np.linalg.inv(bread)
    V = Binv @ meat @ Binv                        # sandwich
    se = np.sqrt(np.diag(V))

    # 连续变量 AME 的常用近似：平均密度 * 系数
    # 二元虚拟变量更精确的做法是 "样本平均差分预测"，此处统一用连续公式便于对照教材表
    ame = phi.mean() * beta

    loglik = float(np.sum(y * np.log(Phi) + (1.0 - y) * np.log(1.0 - Phi)))
    return beta, se, ame, loglik, it + 1


def make_X_demo(d):
    """常数 + age + education + Black + Hispanic（与正文 Table 25.1 类设定一致）"""
    black = (d["race"].values == 2).astype(float)  # race==2：黑人（CPS 编码）
    hisp = d["hisp"].values.astype(float)
    X = np.column_stack([
        np.ones(len(d)),
        d["age"].values.astype(float),
        d["education"].values.astype(float),
        black,
        hisp,
    ])
    names = ["const", "age", "education", "Black", "Hispanic"]
    return X, names


def married_indicator(d):
    """已婚类：marital ∈ {1,2,3}（已婚/丧偶等；与 Hansen 图/表常用口径一致）"""
    return d["marital"].isin([1, 2, 3]).astype(float).values


def print_table(title, names, beta, se, ame, n, ymean, ll, nit):
    print(f"\n=== {title} ===")
    print(f"n = {n},  mean(Y) = {ymean:.4f},  logL = {ll:.1f},  iters = {nit}")
    print(f"{'var':12s} {'coef':>10s} {'SE':>10s} {'AME':>10s}")
    print("-" * 46)
    for i, nm in enumerate(names):
        print(f"{nm:12s} {beta[i]:10.4f} {se[i]:10.4f} {ame[i]:10.4f}")


# 读入 CPS 2009 March
cps = pd.read_stata(ROOT / "cps09mar" / "cps09mar.dta")
print("cps09mar loaded:", cps.shape)
print(cps.columns.tolist())


## Exercise 25.15　男性：工会 membership 的 probit

- 样本：`female == 0`
- $Y = 1\{\texttt{union}=1\}$（本数据中均值约 **2.3%**，极不平衡）
- $X$：常数、age、education、Black、Hispanic


In [ ]:
# ---- 25.15 男性 × 工会 ----
men = cps.loc[cps["female"] == 0].copy()
y_u_m = men["union"].values.astype(float)
X_m, names = make_X_demo(men)

b, se, ame, ll, nit = probit_mle(y_u_m, X_m)
print_table("25.15 Men: Union membership (probit)", names, b, se, ame,
            len(men), y_u_m.mean(), ll, nit)

print("""
解读提示：
- 年龄 AME ≈ +0.0004/年（每年约 0.04 个百分点），效应很小但符号为正。
- 教育系数为负：本样本中更高学历男性参与工会的条件概率略低。
- Hispanic 显著为负；Black 不显著。
- Y 极稀有时，拟合概率整体贴在 0 附近，看 AME 比看 raw 系数更稳妥。
""")


## Exercise 25.16　女性：同上


In [ ]:
# ---- 25.16 女性 × 工会 ----
wom = cps.loc[cps["female"] == 1].copy()
y_u_w = wom["union"].values.astype(float)
X_w, names = make_X_demo(wom)

b, se, ame, ll, nit = probit_mle(y_u_w, X_w)
print_table("25.16 Women: Union membership (probit)", names, b, se, ame,
            len(wom), y_u_w.mean(), ll, nit)

print("""
与男性对比：
- 女性 **education 为正**（男性为负）：高学历女性工会参与率略高。
- 年龄同向、都小；西班牙裔仍为负。
- 参与率同样 ~2%，推断精度有限。
""")


## Exercise 25.17　大学学历女性：婚姻对年龄的二次指数

对照 Figure 25.1：婚姻概率随年龄 **先升后略降**，线性 probit 不够。

$$
P(\mathrm{married}\mid \mathrm{age})
= \Phi\bigl(\beta_0 + \beta_1\,\mathrm{age} + \beta_2\,\mathrm{age}^2/100\bigr).
$$

样本：女性且 `education ≥ 16`。并给出大学男性同规格作对比。


In [ ]:
# ---- 25.17 大学女性：marriage ~ age + age^2/100 ----
cw = wom.loc[wom["education"] >= 16].copy()
y_cw = married_indicator(cw)
age = cw["age"].values.astype(float)
X_cw = np.column_stack([np.ones(len(cw)), age, (age ** 2) / 100.0])
names_q = ["const", "age", "age2/100"]

b, se, ame, ll, nit = probit_mle(y_cw, X_cw)
print_table("25.17 College women: marriage ~ quadratic age", names_q, b, se, ame,
            len(cw), y_cw.mean(), ll, nit)

print("\n预测概率 P(married | age):")
for a in [25, 35, 45, 55, 65]:
    eta = b[0] + b[1] * a + b[2] * (a ** 2) / 100.0
    print(f"  age={a:2d}:  Phi = {norm.cdf(eta):.3f}")

# 峰值年龄：d/da (b0 + b1 a + b2 a^2/100) = 0 ⇒ a* = -50 * b1 / b2
if b[2] < 0:
    a_star = -50.0 * b[1] / b[2]
    print(f"\n指数峰值年龄 a* = -50*b1/b2 ≈ {a_star:.1f} 岁（二次项为负时有意义）")

# ---- 对比：大学男性 ----
cm = men.loc[men["education"] >= 16].copy()
y_cm = married_indicator(cm)
age_m = cm["age"].values.astype(float)
X_cm = np.column_stack([np.ones(len(cm)), age_m, (age_m ** 2) / 100.0])
bm, sem, amem, llm, nitm = probit_mle(y_cm, X_cm)
print_table("Compare: College men (same spec)", names_q, bm, sem, amem,
            len(cm), y_cm.mean(), llm, nitm)
print("\n大学男性预测概率:")
for a in [25, 35, 45, 55, 65]:
    eta = bm[0] + bm[1] * a + bm[2] * (a ** 2) / 100.0
    print(f"  age={a:2d}:  Phi = {norm.cdf(eta):.3f}")

print("""
对比要点：
- 女性峰值更早、高年龄段回落更明显（65 岁约 0.47）。
- 男性中年平台更高更宽（45–55 仍在 0.86–0.87）。
""")


## Exercise 25.18　男性：婚姻 ~ age, education, Black, Hispanic

线性指数 probit（全年龄男性）。注意：全样本线性 age 是对全局的粗糙平均；青年子样本斜率通常更大（见正文 Table 25.1）。


In [ ]:
# ---- 25.18 男性 × 婚姻（线性协变量）----
y_m_m = married_indicator(men)
X_m, names = make_X_demo(men)
b, se, ame, ll, nit = probit_mle(y_m_m, X_m)
print_table("25.18 Men: Marriage (linear index probit)", names, b, se, ame,
            len(men), y_m_m.mean(), ll, nit)

print("""
解读：
- 年龄 AME ≈ +0.97 个百分点/年。
- 教育提高婚姻概率（AME ≈ +1.2 pp / 年教育）。
- 黑人男性显著更低（AME ≈ −15 pp）；西班牙裔与基组接近。
""")


## Exercise 25.19　女性：同 25.18


In [ ]:
# ---- 25.19 女性 × 婚姻 ----
y_m_w = married_indicator(wom)
X_w, names = make_X_demo(wom)
b, se, ame, ll, nit = probit_mle(y_m_w, X_w)
print_table("25.19 Women: Marriage (linear index probit)", names, b, se, ame,
            len(wom), y_m_w.mean(), ll, nit)

print("""
与男性对比：
- 年龄 AME 约为男性一半（~0.47 pp/年）。
- 教育效应相近。
- Black / Hispanic 负效应在女性中更大（种族差距更突出）。
- 精细年龄剖面仍建议用 25.17 的二次设定。
""")


## 理论题速览（25.1–25.14）

完整 step-by-step 证明见 `Hansen_Ch25_Exercises_Solutions.md`。此处只列结论锚点：

| 题 | 结论 |
|:--:|------|
| 25.1 | 翻转 $Y$ ⇒ probit 系数全部反号 |
| 25.2 | $X$ 改成千元 ⇒ 对应 $\beta$ 放大 1000 倍 |
| 25.3 | $e\in\{1-P,-P\}$，$\mathrm{Var}(e\mid X)=P(1-P)$ |
| 25.4 | $\pi(Y\mid X)=G(Z'\beta)$，$Z=(2Y-1)X$ 型 |
| 25.5–6 | logit/probit 的 $h,H$；probit $h=\phi/\Phi$（Mills） |
| 25.7 | score/Hessian；$H(x)>0$ ⇒ 全局凹、MLE 唯一 |
| 25.8–10 | 总体/样本 FOC（logit 化为 $\sum X(Y-\Lambda)=0$） |
| 25.11 | (25.14) 与 logit 下 $h^2=(Y-\Lambda)^2$ |
| 25.12 | NLLS-probit 一致但一般异于 MLE |
| 25.13 | 控制函数/残差纳入后 $\varepsilon\perp Y_2$ |
| 25.14 | 非参下只识别 $m/\sigma$；须规范化尺度 |
